# AmazingHand — Rock, Paper, Scissors

Chọn kernel **Python 3.12**. Chạy Cell 1–8, sau đó Cell 9 để mở game.

Thư viện và model thiếu được tự cài/tải; lần đầu cần Internet, Laya khoảng 808 MB. Restart Kernel nếu cell setup yêu cầu. Cần camera và driver USB khi dùng bàn tay thật.

In [ ]:
from pathlib import Path
import os
import sys
import platform
import importlib
import importlib.util
from importlib.metadata import version, PackageNotFoundError
import atexit
import shlex

os.environ["KERAS_BACKEND"] = "torch"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "PythonExample" / "AmazingHand_RPSGame.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("FAIL — Mở notebook từ thư mục AmazingHand hoặc PythonExample.")
helper_path = str(PROJECT_ROOT / "PythonExample")
if helper_path not in sys.path:
    sys.path.insert(0, helper_path)


if "close_session" in globals():
    close_session()
resources = {}
def close_session():
    errors = []
    for key in ("game", "robot", "camera", "recognizer", "decision_engine", "display"):
        resource = resources.pop(key, None)
        if resource is not None:
            try:
                resource.close()
            except Exception as exc:
                errors.append(f"{key}: {exc}")
    if errors:
        print("Lỗi giải phóng tài nguyên:", errors)
if "_rps_exit_callback" in globals():
    atexit.unregister(_rps_exit_callback)
_rps_exit_callback = close_session
atexit.register(_rps_exit_callback)

print("Python:", platform.python_version(), "—", sys.executable)
print("Hệ điều hành:", platform.platform())
print("PASS" if sys.version_info >= (3, 11) else "FAIL", "— Python 3.11–3.12; kernel khuyến nghị 3.12")
requirements = {
    "numpy": "numpy==1.26.4", "scipy": "scipy==1.16.1",
    "keras": "keras==3.15.1", "h5py": "h5py>=3.11,<4", "cv2": "opencv-contrib-python==4.11.0.86",
    "torch": "torch>=2", "transformers": "transformers==4.57.6",
    "laya": "laya==0.4.3", "huggingface_hub": "huggingface_hub>=0.20",
    "safetensors": "safetensors>=0.4", "rustypot": "rustypot==1.11.0",
    "serial": "pyserial>=3.5,<4", "IPython": "ipython",
    "ipywidgets": "ipywidgets>=8,<9", "jupyter_ui_poll": "jupyter-ui-poll>=0.2,<1",
}
from AmazingHand_Setup import ensure_dependencies
ensure_dependencies(PROJECT_ROOT / "requirements-rps-game.txt")
missing = []
environment_errors = ["Package chưa cài: " + item for item in missing]
environment_ok = not missing
for module in requirements:
    try:
        imported = importlib.import_module(module)
        print("PASS —", module, getattr(imported, "__version__", "đã import"))
    except Exception as exc:
        environment_ok = False
        environment_errors.append(f"{module}: {type(exc).__name__}: {exc}")
        print("FAIL —", module, type(exc).__name__, str(exc))

if environment_ok:
    import torch
    torch.set_num_threads(min(8, os.cpu_count() or 1))
    from transformers.modeling_utils import no_init_weights
    import keras
    print("PASS — CPU; CUDA:", torch.cuda.is_available())
    if keras.backend.backend() != "torch":
        raise RuntimeError("Keras backend chưa đúng. Restart Kernel rồi chạy lại Cell 1.")
    print("PASS — Keras/PyTorch; model RPS:", (PROJECT_ROOT / "models/rps.h5").is_file())
    for name in ("AmazingHand_HardwareDemo", "AmazingHand_Motion", "AmazingHand_GripCycles",
                 "AmazingHand_VSign", "AmazingHand_Notebook", "AmazingHand_LayaRPS", "AmazingHand_RPSGame"):
        try:
            module = importlib.import_module(name)
            importlib.reload(module)
            print("PASS —", name)
        except Exception as exc:
            environment_ok = False
            environment_errors.append(f"{name}: {type(exc).__name__}: {exc}")
            print("FAIL —", name, str(exc))
    import cv2
    probe = cv2.VideoCapture(0, cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY)
    try:
        ok, _ = probe.read() if probe.isOpened() else (False, None)
        print("PASS" if ok else "FAIL", "— Camera 0; có thể đổi camera_id tại Cell 2.")
    finally:
        probe.release()
    from AmazingHand_RPSGame import GameNotReady
    game = GameNotReady()
if not environment_ok:
    raise RuntimeError("FAIL — Lỗi dependency/module:\n" + "\n".join(environment_errors) + "\nKernel: " + sys.executable)
print("PASS — Môi trường sẵn sàng. Tiếp theo: Cell 2.")

## Cell 2 — Cấu hình

- `ROBOT_ENABLED=False`: chạy không cần bàn tay; `PORT="auto"`: tự chọn USB serial duy nhất.
- `speed_deg_s`: tốc độ khớp (°/giây); `snapshot_delay_seconds`: độ trễ chụp sau RA TAY.
- `roi`: vùng tay người theo tỷ lệ 0–1; đặt robot ngoài vùng này.
- `laya_exploration`: tăng để nước robot đa dạng hơn.

Sau khi đổi tốc độ, chạy lại Cell 2, 6 và 8.

In [ ]:
from AmazingHand_RPSGame import GameConfig
from AmazingHand_Setup import choose_serial_port

ROBOT_ENABLED = True
PORT = "auto"

config = GameConfig(
    camera_id=0,
    camera_width=1280,
    camera_height=720,
    camera_fps=30,
    window_width=1920,
    window_height=1080,
    target_wins=3,
    show_confirmation_buttons=False,
    laya_exploration=0.30,
    require_confirmation=True,
    confidence_threshold=0.75,
    smoothing_window=5,
    minimum_agreement=4,
    countdown_seconds=3,
    snapshot_delay_seconds=0.1, # Giây sau RA TAY
    rps_model_path="models/rps.h5",
    rps_class_names=("paper", "rock", "scissors"),
    rps_pixel_scale=1/255,
    prediction_margin=0.15,
    minimum_roi_std=8,
    robot_enabled=ROBOT_ENABLED,
    roi=(0.03, 0.05, 0.58, 0.95),
    preview_seconds=12,
    display_backend="auto",
    port=choose_serial_port(PORT) if ROBOT_ENABLED else PORT,
    speed_deg_s=220, # °/giây
    reset_move="rock",
    open_extensions=(70, 60, 37, 35),
    close_flexions=(92, 90, 90, 90),
)
config.validate()
print("PASS — Cấu hình:", config)

## Cell 3 — Model RPS

Nạp `models/rps.h5` một lần, kiểm tra checksum và khởi tạo inference local.

In [ ]:
from AmazingHand_RPSGame import RPSModelSession
from AmazingHand_Setup import ensure_rps_model

if resources.get("recognizer") is None or resources["recognizer"].closed:
    resources["recognizer"] = RPSModelSession(ensure_rps_model(PROJECT_ROOT, config.rps_model_path), config)
recognizer_session = resources["recognizer"]
print("PASS — RPS H5 local: RGB 150×150 /255; nhãn:", config.rps_class_names)

## Cell 4 — Kiểm tra camera

Đặt một tay trong ROI, nền đơn giản, đủ sáng. Thử ROCK, PAPER và SCISSORS. Preview tự kết thúc; Q/Esc để thoát.

In [ ]:
from AmazingHand_RPSGame import CameraStream, GestureDetector, GameDisplay, preview_camera

if resources.get("game") is not None:
    resources.pop("game").close(keep_models=True)
if resources.get("camera") is not None:
    resources["camera"].close()
camera = CameraStream(config).open()
resources["camera"] = camera
preview_detector = GestureDetector(recognizer_session, config)
preview_display = GameDisplay(config)
preview_gestures = preview_camera(camera, preview_detector, preview_display, config.preview_seconds)
print("PASS — Camera được giữ mở; tiếp theo Cell 5.")

## Cell 5 — Nhận diện

Mỗi ván phân loại một ảnh đã chụp. Ảnh RGB 150×150, chia 255; nhãn `paper / rock / scissors`.

Model có thể nhầm hoặc trả UNKNOWN; phím 1/2/3 xác nhận nước thực tế.

In [ ]:
from AmazingHand_RPSGame import GestureDetector

detector = GestureDetector(recognizer_session, config)
print("PASS — RPS H5, nhãn paper/rock/scissors, phân loại một ảnh/ván, lọc confidence/margin.")

## Cell 6 — Laya và bàn tay

Laya chọn nước từ lịch sử trước khi xem ảnh hiện tại; ván đầu dùng lịch sử rỗng. Mặc định trộn 30% khám phá để đa dạng nước đi.

Model và serial được dùng lại trong phiên. Đóng ứng dụng khác đang giữ cổng bàn tay.

In [ ]:
import time
import torch
import laya
from AmazingHand_LayaRPS import ensure_model, MODEL_REVISION
from AmazingHand_RPSGame import LayaDecisionEngine, AmazingHandRobotAdapter

if resources.get("decision_engine") is None or resources["decision_engine"].closed:
    model_path = ensure_model(PROJECT_ROOT / "models" / "laya-english", offline=config.offline)
    device = ("cuda" if torch.cuda.is_available() else "cpu") if config.device == "auto" else config.device
    if device == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("FAIL — PyTorch chưa có CUDA; chọn config.device='cpu'.")
    if device == "cpu":
        torch.set_num_threads(min(8, os.cpu_count() or 1))
    started = time.perf_counter()
    previous_offline = os.environ.get("HF_HUB_OFFLINE")
    os.environ["HF_HUB_OFFLINE"] = "1"
    try:
        agent = laya.load(str(model_path), device=device, fast=False, compile=False)
    finally:
        if previous_offline is None:
            os.environ.pop("HF_HUB_OFFLINE", None)
        else:
            os.environ["HF_HUB_OFFLINE"] = previous_offline
    resources["decision_engine"] = LayaDecisionEngine(agent, config=config)
    print(f"PASS — Laya nạp trong {time.perf_counter()-started:.2f}s, device={agent.device}")
decision_engine = resources["decision_engine"]
if resources.get("robot") is not None:
    resources["robot"].close()
robot_controller = AmazingHandRobotAdapter(config).connect()
resources["robot"] = robot_controller
print("PASS — choose_move(history), execute_move(move), reset_pose() sẵn sàng.")

## Cell 7 — Luồng chơi

**Countdown → RA TAY → chụp ảnh → nhận diện → chấm điểm.**

Xác nhận bằng 1/2/3 ngay khi hết countdown, không cần chờ ảnh hoặc motor. Lựa chọn được giữ lại; chỉ chấm điểm sau khi motor hoàn tất. Nước đã xác nhận được lưu cho Laya ở ván sau.

In [ ]:
from AmazingHand_RPSGame import RPSGame, GameDisplay

print("PASS — Bộ điều khiển game đã sẵn sàng. Chưa tự chạy round.")

## Cell 8 — Khởi tạo phiên

Chuẩn bị camera, model và bàn tay. Cell 9 mở game; **R** reset cho người chơi mới. `close_session()` giải phóng toàn bộ tài nguyên.

In [ ]:
if resources.get("game") is not None:
    resources.pop("game").close(keep_models=True)
if recognizer_session.closed or decision_engine.closed:
    raise RuntimeError("Model đã đóng. Chạy lại Cell 3, 5, 6 rồi Cell 8.")
camera.open()
robot_controller.connect()
camera.get_frame()
display = GameDisplay(config)
resources["display"] = display
game = RPSGame(
    config=config, camera=camera, detector=detector,
    decision_engine=decision_engine, robot_controller=robot_controller,
    display=display, log_path=PROJECT_ROOT / config.log_path,
)
resources["game"] = game
print("GAME READY — Chạy cell cuối để bắt đầu trận chạm 3 điểm thắng.")

### Thử ROCK

Co cả 4 ngón. Chạy sau Cell 8, khi game đã đóng; giữ tư thế sau khi hoàn tất.

In [ ]:
_manual_robot = globals().get("robot_controller")
if _manual_robot is None or _manual_robot.closed:
    print("Chạy Cell 6 và Cell 8 để khởi tạo bàn tay trước.")
elif "game" in globals() and (getattr(game, "busy", False)
        or getattr(getattr(game, "display", None), "app_active", False)):
    print("Thoát cửa sổ game trước khi thử tư thế riêng.")
else:
    print("[HAND] ROCK: Co cả 4 ngón.", flush=True)
    _pose_result = _manual_robot.execute_move("rock")
    print("[HAND] Mô phỏng: không gửi lệnh motor." if _pose_result["simulated"]
          else "[HAND] Đã xác nhận đạt tư thế ROCK, giữ torque.", flush=True)

### Thử PAPER

Duỗi cả 4 ngón. Dùng kết nối và tốc độ đã cấu hình; không thay đổi điểm hoặc lịch sử.

In [ ]:
_manual_robot = globals().get("robot_controller")
if _manual_robot is None or _manual_robot.closed:
    print("Chạy Cell 6 và Cell 8 để khởi tạo bàn tay trước.")
elif "game" in globals() and (getattr(game, "busy", False)
        or getattr(getattr(game, "display", None), "app_active", False)):
    print("Thoát cửa sổ game trước khi thử tư thế riêng.")
else:
    print("[HAND] PAPER: Duỗi cả 4 ngón.", flush=True)
    _pose_result = _manual_robot.execute_move("paper")
    print("[HAND] Mô phỏng: không gửi lệnh motor." if _pose_result["simulated"]
          else "[HAND] Đã xác nhận đạt tư thế PAPER, giữ torque.", flush=True)

### Thử SCISSORS

Duỗi index và middle; co ring và thumb. Chạy sau Cell 8, khi game đã đóng.

In [ ]:
_manual_robot = globals().get("robot_controller")
if _manual_robot is None or _manual_robot.closed:
    print("Chạy Cell 6 và Cell 8 để khởi tạo bàn tay trước.")
elif "game" in globals() and (getattr(game, "busy", False)
        or getattr(getattr(game, "display", None), "app_active", False)):
    print("Thoát cửa sổ game trước khi thử tư thế riêng.")
else:
    print("[HAND] SCISSORS: Duỗi index và middle; co ring và thumb.", flush=True)
    _pose_result = _manual_robot.execute_move("scissors")
    print("[HAND] Mô phỏng: không gửi lệnh motor." if _pose_result["simulated"]
          else "[HAND] Đã xác nhận đạt tư thế SCISSORS, giữ torque.", flush=True)

## Cell 9 — Mở game

Chạy một lần, thao tác trong cửa sổ 1920×1080:

- **Enter:** bắt đầu, sang ván hoặc chuẩn bị người chơi mới.
- **1 / 2 / 3:** xác nhận ROCK / PAPER / SCISSORS ngay khi hết countdown.
- **R:** reset điểm và lịch sử giữa các ván.
- **Q / Esc:** thoát; nếu phiên bị khóa, chạy lại Cell 8.

**Ai chạm 3 điểm trước thắng; hòa chơi tiếp.** Tư thế chờ: co cả 4 ngón. Chọn cửa sổ game để nhận phím; log chi tiết lưu vào file.

In [ ]:
if "game" not in globals():
    print("Chạy Cell 8 để khởi tạo game trước.")
else:
    game.run_app()